In [ ]:
# 📦 Install LangChain v1 and the integration packages used in this notebook
# Pinned to the version requested: langchain==1.3.11
#%pip install -q "langchain==1.3.11" langchain-openai langchain-tavily langgraph python-dotenv

As of **LangChain v1**, `initialize_agent` and `AgentType` are deprecated and have moved to the `langchain-classic` package. The new standard for building agents is [`create_agent`](https://docs.langchain.com/oss/python/releases/langchain-v1#create_agent) — a single, unified, middleware-driven agent constructor built on LangGraph, replacing `langgraph.prebuilt.create_react_agent` as well as every `AgentType` variant.

Each section below reproduces the *intent* of one of the original 7 agent types using `create_agent`, plus a bonus section on **middleware** — the core new capability in v1 that replaces most of what the old `AgentType` variants used to hard-code.

Reference: <https://docs.langchain.com/oss/python/releases/langchain-v1#create_agent>


## Migration cheat-sheet: `AgentType` → LangChain v1

| # | Legacy `AgentType` (pre-v1) | What it did | v1 equivalent with `create_agent` |
|---|---|---|---|
| 1 | `ZERO_SHOT_REACT_DESCRIPTION` | Single-shot ReAct loop, no memory | `create_agent(model, tools)` — the default agent loop |
| 2 | `CONVERSATIONAL_REACT_DESCRIPTION` | ReAct + `ConversationBufferMemory` | `create_agent(..., checkpointer=InMemorySaver())` + `thread_id` |
| 3 | `CHAT_CONVERSATIONAL_REACT_DESCRIPTION` | Chat-model ReAct + memory | Same as above — chat models are the default everywhere now |
| 4 | `STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION` | Multi-argument tools via a structured schema | Any `@tool`-decorated function passed straight to `create_agent` |
| 5 | `OPENAI_FUNCTIONS` | Function-calling instead of text-based ReAct parsing | Native tool calling — `create_agent` always uses the model's function/tool-calling API when available |
| 6 | `OPENAI_MULTI_FUNCTIONS` | Multiple function calls in a single step | Native parallel tool calling — no special agent type needed |
| 7 | `SELF_ASK_WITH_SEARCH` | Explicit `Follow up: ... / Intermediate answer: ...` prompting pattern | A `system_prompt` that encodes the self-ask pattern, combined with a search tool |

Other things that moved or were renamed:

- `langchain.chat_models.ChatOpenAI` → `langchain_openai.ChatOpenAI` (or `langchain.chat_models.init_chat_model`)
- `langchain.chains.LLMChain` → moved to `langchain-classic`; prefer a plain `@tool` function or an LCEL chain
- `langchain.memory.ConversationBufferMemory` → replaced by LangGraph **checkpointers** (`InMemorySaver`, `PostgresSaver`, ...)
- `langchain.tools.TavilySearchResults` (community, deprecated) → `langchain_tavily.TavilySearch`


In [1]:

import os
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver

# Load API keys
load_dotenv(".env", override=True)

google_api_key = os.getenv("GEMINI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

if not google_api_key:
    raise ValueError("GEMINI_API_KEY not found in .env")

if not tavily_api_key:
    raise ValueError("TAVILY_API_KEY not found in .env")

# Gemini model
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=google_api_key
)

# Simple QA tool
@tool
def simple_qa(question: str) -> str:
    """Answer a question clearly and directly."""
    response = model.invoke(
        f"Answer clearly: {question}"
    )
    return response.content

# Tavily web search
web_search = TavilySearch(max_results=3)

tools = [simple_qa, web_search]

# Test Gemini
response = model.invoke("Explain Generative AI in simple words.")
print(response.content)


Imagine a computer program that isn't just good at following instructions or crunching numbers, but can actually **create brand new things** that have never existed before.

Here's the simple breakdown:

1.  **It Learns:** Think of it like a very talented student or artist who has studied millions of examples of something.
    *   If it's a text-generating AI, it's read countless books, articles, and conversations.
    *   If it's an image-generating AI, it's seen millions of pictures.
    *   If it's a music-generating AI, it's listened to tons of songs.

2.  **It Understands Patterns:** Because it's absorbed so much information, it starts to understand the patterns, styles, and rules of what makes things "look right," "sound right," or "make sense."

3.  **It Generates:** You then give it a "prompt" – basically, an idea or a request (e.g., "Write a poem about a space cat," or "Draw a futuristic city at sunset"). Using all its learned knowledge, it then **generates something completel

## 1) Basic agent (replaces `ZERO_SHOT_REACT_DESCRIPTION`)

The zero-shot ReAct agent was the "simple + reliable default" in the legacy API — no memory, single-pass reasoning + tool use. In v1 this is simply `create_agent` with no `checkpointer` and no extra middleware.

In [2]:

# 1. Basic Gemini Agent with Tavily Search

import os
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver

# Load API keys
load_dotenv(".env", override=True)

# Check Tavily API key
tavily_api_key = os.getenv("TAVILY_API_KEY")

if not tavily_api_key:
    raise ValueError("TAVILY_API_KEY not found in .env")

# Initialize Tavily Search
web_search = TavilySearch(
    max_results=3,
    tavily_api_key=tavily_api_key
)

# Define tools
tools = [web_search]

# Initialize memory
memory = InMemorySaver()

# Create Gemini Agent
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "You are a helpful AI assistant. "
        "Answer clearly and accurately. "
        "Use web search when current or factual information is needed."
    ),
    checkpointer=memory
)

# Run Agent
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Summarize LangChain in 2 lines, then tell me who created it."
            }
        ]
    },
    config={
        "configurable": {
            "thread_id": "1"
        }
    }
)

# Extract final response
final_message = result["messages"][-1]

# Print clean text
if isinstance(final_message.content, str):
    print(final_message.content)
else:
    for block in final_message.content:
        if isinstance(block, dict) and block.get("type") == "text":
            print(block["text"])


LangChain is an open-source framework that simplifies the development of applications powered by large language models (LLMs). It provides tools to connect LLMs with external data and computation, enabling the creation of complex, context-aware AI agents.

It was created by Harrison Chase.


In [ ]:
Component	Purpose
Gemini model	Generates answers
Tavily Search	Searches the web
create_agent	Creates the AI agent
InMemorySaver	Stores conversation state
thread_id	Identifies the conversation
result	Contains agent responses

## 2) Agent with conversational memory (replaces `CONVERSATIONAL_REACT_DESCRIPTION`)

`ConversationBufferMemory` is gone. Short-term memory in v1 is handled by a **checkpointer** attached to the agent, addressed by a `thread_id`. Every `invoke()` call on the same thread automatically sees the full prior message history — no manual memory object to wire up.

In [14]:
# 2) Agent with conversational memory (replaces `CONVERSATIONAL_REACT_DESCRIPTION`)
checkpointer = InMemorySaver()

memory_agent = create_agent(
    model=model,
    tools=tools,
    checkpointer=checkpointer,
)

thread_config = {"configurable": {"thread_id": "agent-types-demo-1"}}

result = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "Summarize LangChain in 2 lines, then tell me who created it."}]},
    thread_config,
)
print(result["messages"][-1].content)

[{'type': 'text', 'text': 'LangChain is an open-source framework for developing applications powered by large language models (LLMs), offering composable tools to connect various components like document loaders, text splitters, and LLM wrappers. It simplifies the creation of complex agent systems.\n\nLangChain was created by Harrison Chase.', 'extras': {'signature': 'CtgDAWkUfRPWOcd1unG4kbVEbi10/HIj7dTYqsLznsMYFF1giWsJmvU1yZGdPgcPpeGw6UxJYBqq25ZZqqTA6Y10xt526QfDyXr5LW262GVtQU0/QzY72ZgavbpdqpuSbeo5WrMeqvath/v7RXmKCdYe6WdRdlfcDb2s8rCGs98Su71rLKoUK0AwFtHCBTQQAdw99JJpzCzkjx83dy2XvGGWmhWXtM0e/eOCTnNSeCMe4a6KWjp/U9YJ066V8bQaaFlVo7L98ErGreL2/pMZTRMAdMaNPgLL6OgO06QQiweMiRioHITpAiDTvh/KTcWeHG4k8OrcfCENB3bWLebxgkDAQ06+g9+2htgMCj84whX31Ygwjb+qWBRRofB31o+z1Zr37c2jioADKj1OBQ9c7GiNc46j6aE2PgtRDezDEbygQSevpipTx31Q/EEajN7dMDxjsTYdZJaqwt2kT3mFb+1vpIf7PFkc6Cqe3GfgBl+4qhyX4jaAhDNiGej4BEBv9iTYqRljqfnwJDOs5OiFsJ0b71GmcCDsAb7YttAsOQtW9ET7qEGH1WkEzQfUIRh670wdokvCVMbuj2FRtWN5KnhwJgKEqru04K8Pxx1FnZwORBgJPT3yP

In [15]:

# 2) Agent with conversational memory (replaces `CONVERSATIONAL_REACT_DESCRIPTION`)

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# Create memory checkpointer
checkpointer = InMemorySaver()

# Create conversational memory agent
memory_agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "You are a helpful AI assistant. "
        "Answer clearly and accurately. "
        "Use Tavily web search whenever current information is required."
    ),
    checkpointer=checkpointer
)

# Define conversation thread
thread_config = {
    "configurable": {
        "thread_id": "agent-types-demo-1"
    }
}

# First conversation
result = memory_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Summarize LangChain in 2 lines, then tell me who created it."
            }
        ]
    },
    config=thread_config
)

# Extract and print response
final_message = result["messages"][-1]

if isinstance(final_message.content, str):
    print(final_message.content)
else:
    for block in final_message.content:
        if isinstance(block, dict) and block.get("type") == "text":
            print(block["text"])


LangChain is an open-source framework designed to simplify the creation of applications using large language models (LLMs) by providing modular components that can be chained together. It was created by Harrison Chase.


## 3) Multi-turn chat agent (replaces `CHAT_CONVERSATIONAL_REACT_DESCRIPTION`)

`CHAT_CONVERSATIONAL_REACT_DESCRIPTION` mainly existed to point the conversational agent at a chat model instead of a plain text-completion LLM. In v1 every model is a chat model, so that distinction disappears — the memory-backed agent above already gives us multi-turn chat. Here we simply continue the **same thread** to show a follow-up question that depends on prior context:

In [16]:
## 3) Multi-turn chat agent (replaces `CHAT_CONVERSATIONAL_REACT_DESCRIPTION`)
follow_up = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "Now plan a 3-step study path for what I just asked about."}]},
    thread_config,  # same thread_id -> agent remembers the prior turn
)
print(follow_up["messages"][-1].content)

[{'type': 'text', 'text': "Here's a 3-step study path for LangChain:\n\n1.  **Grasp the Fundamentals:** Learn about LangChain's core modules (models, prompts, chains, agents, retrievers) and their individual roles in building LLM applications.\n2.  **Explore Use Cases and Examples:** Investigate how LangChain is used to create various applications like chatbots, question-answering systems, and data augmentation tools.\n3.  **Build a Basic Application:** Get hands-on by developing a simple LLM application using LangChain, such as a basic conversational agent or a document summarizer.", 'extras': {'signature': 'CpIGAWkUfRMrNNmAZy+Yi8D7jY88+Hh3X5FkLC5Zl8lqse26nGgm7YNmMyolHv0m8Sq8bnMkA2RCEc6BJIviTk4vFKidlsiVNpLS9J0krNB1cHEkPO4Q9azgeIrhKWPOUIvzA1QCnzqA/vFnkxvt8Q2iT0cSNObsIpvy8ub/GW7Xu2L+y/A2vPN/cQ3XsP/06yaFUHmxp6sKTVpmp/wzL54XiXMc4amNDFkv5JIJnLz3HaOhS2GDO+94g9cAzjs/iCtBA4wPvhwacaopvEitsWSfiiJEB9j+vIxjUjqrkrq/4m5EQxOj7gVezAWVOMLfIo5uM73oUpEVRzdl2ysWKBOIvgArh9OOQGj+B7h2Fy1WFRlHHhCK+WkZGoAWItS

In [17]:

# 3. Multi-turn Chat Agent

# Continue the previous conversation
follow_up = memory_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Now plan a 3-step study path for what I just asked about."
            }
        ]
    },
    config=thread_config
)

# Extract final response
final_message = follow_up["messages"][-1]

# Print clean text
if isinstance(final_message.content, str):
    print(final_message.content)
else:
    for block in final_message.content:
        if isinstance(block, dict) and block.get("type") == "text":
            print(block["text"])


Here's a 3-step study path for planning a 3-step study path:

1.  **Define the Learning Objective:** Clearly identify the specific skill or knowledge the study path aims to achieve.
2.  **Deconstruct the Topic:** Break down the subject into its fundamental components and logical progression of learning.
3.  **Sequence and Elaborate:** Arrange these components into a sequential flow, providing a brief description or action for each step.


In [ ]:
Component	Purpose
InMemorySaver()	Saves conversation checkpoints
thread_id	Identifies the conversation
memory_agent	Maintains conversation context
invoke()	Sends a new message

## 4) Structured, multi-argument tools (replaces `STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION`)

`StructuredTool` + a Pydantic `args_schema` used to require its own dedicated `AgentType`. In v1, **every** tool passed to `create_agent` supports multi-argument, typed, structured calling out of the box — no special agent type required. The `@tool` decorator infers the schema straight from your function's type hints and docstring.

In [18]:

# 4. Structured, Multi-Argument Tools

from langchain.tools import tool
from langchain.agents import create_agent

# Define a structured tool
@tool
def title_maker(topic: str, tone: str = "concise") -> str:
    """
    Generate a title based on a topic and optional tone.
    Tone can be concise, playful, or formal.
    """
    return f"{tone.title()} Title: {topic} in Practice"


# Create structured agent
structured_agent = create_agent(
    model=model,
    tools=[title_maker],
    system_prompt=(
        "You are a creative title generator. "
        "Use the title_maker tool to generate titles."
    )
)

# Run the agent
result = structured_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Make a friendly title about LangGraph tutorials."
            }
        ]
    }
)

# Extract final response
final_message = result["messages"][-1]

# Print clean output
if isinstance(final_message.content, str):
    print(final_message.content)
else:
    for block in final_message.content:
        if isinstance(block, dict) and block.get("type") == "text":
            print(block["text"])


Here's a playful title for LangGraph tutorials: "LangGraph tutorials in Practice"


## 5) Native function / tool calling (replaces `OPENAI_FUNCTIONS`)

`OPENAI_FUNCTIONS` opted an agent into OpenAI's function-calling schema instead of brittle text-based ReAct parsing. `create_agent` **always** uses the model's native tool-calling API when the model supports it — there's nothing extra to configure. Streaming the run lets us inspect the intermediate `AIMessage.tool_calls` to see the function-calling schema in action:

In [21]:

# 5. Native Function / Tool Calling

# Use the existing agent created in Section 1
# It already contains Gemini model and Tavily Search tool.

# 5. Native Function / Tool Calling
# Gemini + LangChain + Tavily Search

# Thread configuration (required for checkpointer)
thread_config = {
    "configurable": {
        "thread_id": "native-tool-demo-1"
    }
}

# Stream agent execution
for step in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "Search the web for LangGraph documentation and give me 3 bullet points."
            }
        ]
    },
    config=thread_config,
    stream_mode="values"
):

    # Get latest message
    last = step["messages"][-1]

    # Check tool calls
    if getattr(last, "tool_calls", None):
        print(
            "Tool call(s):",
            [tc["name"] for tc in last.tool_calls]
        )

    # Print final answer
    elif last.content:

        print("\nFinal answer:")

        if isinstance(last.content, str):
            print(last.content)

        else:
            for block in last.content:
                if isinstance(block, dict) and block.get("type") == "text":
                    print(block["text"])



Final answer:
Search the web for LangGraph documentation and give me 3 bullet points.
Tool call(s): ['tavily_search']

Final answer:
{"query": "LangGraph documentation", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.scaler.com/topics/langgraph-doc-explained", "title": "LangGraph Docs Explained | The 20% You’ll Use 80% of the Time", "content": "| Resource | Where | What it's for | When to use it |\n ---  --- |\n| Docs by LangChain | docs.langchain.com | Conceptual guides + how-tos | Your daily driver start here |\n| API Reference | reference.langchain.com | Exact signatures of every class/method | When a guide isn't enough |\n| GitHub repo | github.com/langchain-ai/langgraph | Source, issues, examples/, release notes | Stuck, upgrading, or hunting examples | [...] The LangGraph documentation stops being intimidating the moment it has a map: docs.langchain.com for concepts, reference.langchain.com for signatures, GitHub for source and exampl

Code	Meaning
agent.stream()	Runs agent with streaming updates
stream_mode="values"	Returns the full agent state after each step
tool_calls	Shows requested tool calls
tc["name"]	Gets the tool name
last.content	Gets the message content

## 6) Parallel tool calls (replaces `OPENAI_MULTI_FUNCTIONS`)

`OPENAI_MULTI_FUNCTIONS` let the agent call more than one tool within a single step. This is native behavior in v1 — when a prompt needs several tools at once, the model can emit multiple `tool_calls` on a single `AIMessage`, and `create_agent` executes them together, no special agent type needed.

In [3]:

# 6. Parallel Tool Calls
# Gemini + LangChain + Tavily

from langchain.tools import tool
from langchain.agents import create_agent
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver

# Tool 1: Simple QA
@tool
def simple_qa(question: str) -> str:
    """Answer a question clearly and directly."""

    response = model.invoke(
        f"Answer clearly in one sentence: {question}"
    )

    if isinstance(response.content, str):
        return response.content

    return " ".join(
        block["text"]
        for block in response.content
        if isinstance(block, dict) and block.get("type") == "text"
    )


# Tool 2: Web Search
web_search = TavilySearch(max_results=3)

# Add both tools
tools = [simple_qa, web_search]

# Create checkpointer
checkpointer = InMemorySaver()

# Create agent with both tools
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "You are a helpful AI assistant. "
        "When the user requests two independent tasks using "
        "simple_qa and web_search, call both tools independently. "
        "Use both results to prepare the final answer."
    ),
    checkpointer=checkpointer
)

# Thread configuration
thread_config = {
    "configurable": {
        "thread_id": "parallel-tool-demo-1"
    }
}

# Invoke agent
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Use the simple_qa tool to explain what LangChain "
                    "is in one sentence, AND separately use web_search "
                    "to find the latest LangChain release version."
                )
            }
        ]
    },
    config=thread_config
)

# Display tool calls
for msg in result["messages"]:

    if getattr(msg, "tool_calls", None):
        print(
            f"{len(msg.tool_calls)} tool call(s) in this step:",
            [tc["name"] for tc in msg.tool_calls]
        )

# Extract final answer
final_message = result["messages"][-1]

print("\nFinal answer:\n")

if isinstance(final_message.content, str):
    print(final_message.content)

else:
    for block in final_message.content:
        if isinstance(block, dict) and block.get("type") == "text":
            print(block["text"])


2 tool call(s) in this step: ['simple_qa', 'tavily_search']

Final answer:

LangChain is a framework designed to simplify the development of applications powered by large language models by providing tools to connect them with external data sources and other computational steps.

The latest LangChain release version appears to be 1.5.2, according to the official GitHub releases page. However, some search results show an incorrect future date (November 2026) associated with this release.


In [4]:

# 6. Parallel Tool Calls
# Gemini + LangChain + Tavily

import os
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver

# Load API keys
load_dotenv(".env", override=True)

# Tool 1: Simple QA
@tool
def simple_qa(question: str) -> str:
    """Return the question for the main Gemini agent to answer."""
    return (
        "Please answer this question directly using your own knowledge: "
        + question
    )

# Tool 2: Tavily Search
web_search = TavilySearch(max_results=3)

tools = [simple_qa, web_search]

# Create memory
checkpointer = InMemorySaver()

# Create Gemini agent
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "You are a helpful AI assistant. "
        "Use simple_qa for general explanations. "
        "Use web_search for current information. "
        "When asked to use both tools, call both independently."
    ),
    checkpointer=checkpointer
)

# Thread configuration
thread_config = {
    "configurable": {
        "thread_id": "parallel-tool-demo-2"
    }
}

# Run agent
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Use simple_qa to explain LangChain in one sentence, "
                    "and use web_search to find the latest LangChain release version."
                )
            }
        ]
    },
    config=thread_config
)

# Display tool calls
for msg in result["messages"]:
    if getattr(msg, "tool_calls", None):
        print(
            "Tool calls:",
            [tc["name"] for tc in msg.tool_calls]
        )

# Display final answer
final_message = result["messages"][-1]

print("\nFinal Answer:\n")

if isinstance(final_message.content, str):
    print(final_message.content)
else:
    for block in final_message.content:
        if isinstance(block, dict) and block.get("type") == "text":
            print(block["text"])


Tool calls: ['simple_qa', 'tavily_search']

Final Answer:

LangChain is a framework designed to simplify the creation of applications powered by large language models. The latest stable release version of LangChain is 0.1.16, released on April 11, 2024.


## 7) Self-ask with search (replaces `SELF_ASK_WITH_SEARCH`)

The legacy self-ask agent forced a strict `Follow up: ... / Intermediate answer: ...` text format and required a search tool literally named `"Intermediate Answer"`. In v1 there's no dedicated agent type for this — instead we encode the same *reasoning pattern* in the `system_prompt` and let the model decide when to break a question into sub-questions before answering.

In [5]:
## 7) Self-ask with search (replaces `SELF_ASK_WITH_SEARCH`)
self_ask_prompt = """You are a careful research assistant that uses the self-ask technique:
1. Decide if the question needs to be broken into simpler follow-up questions.
2. For each follow-up question, use the web_search tool to find the answer.
3. Once you have every intermediate answer you need, combine them into one final answer.
Always ground factual claims in your search results."""

self_ask_agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=self_ask_prompt,
)

result = self_ask_agent.invoke(
    {"messages": [{"role": "user", "content": "When was the first LangChain release and who founded it?"}]}
)
print(result["messages"][-1].content)

[{'type': 'text', 'text': 'The first LangChain release was in October 2022, with the initial installable package (`langchain` 0.0.1 on PyPI) becoming available on October 25, 2022. LangChain was founded by Harrison Chase, who started it as an open-source project. He later co-founded the company LangChain with Ankush Gola in early 2023.', 'extras': {'signature': 'CqkDAWkUfRM7STMbtgDFRvviGtKq6YKP4X1IUlAU0SfGGDsqnSt6wiWXEJtqu2PkcxY4jfFWLQYMHHHgwySTN3VjlZxlLCD0cyfrfXoo7AmljzBcvre2b05kRqxmYXzmnIgVUUItOzp2UUohkC+8LMl1VV9lEj3nVI8tbyEycFSUKkviPidpcEaithIORCrIkjxiwpjvCSRRKtmAAcL3K61TNnRdxMHfKbdJDkfj3Nou80YfcD5F2KE8035BZM6Qp/bWMpHQW/8H5p3P5RXBVpZk/CmXQ86favX6dTo8fAwJIqvCcV8GGZ14aVnoJXdrE/vjXk1c+1lVeDr+ellSnhFrRQdF067av2Yu6QTa98sLZNDCWKHO5ep/flii3ekK0cBAzxEx+mEH/onj88ZNNt5/cY5pTABLkxOdBz+Sj8KDuzfwheNOw0Weznrj4SqnfmMrp+4fdJmRLBHoJP5YXZePEX49kv9+Q9InxBxcgR2/F5dleP7pQv2Y6BbJg4uEyrBTN8wbV5U7VZavGT7mNOeYGIeVyFybH6xSgWmz7L//S/CXxKuXnPOxvXM='}}]


## Bonus: middleware — the real replacement for most `AgentType` behavior

In the legacy API, every new "flavor" of agent behavior (parsing-error recovery, memory summarization, human approval, etc.) needed its own `AgentType`. In v1, that customization surface is now **middleware**: composable hooks that wrap the model call, wrap each tool call, or run before/after an agent step.

A few prebuilt/custom middlewares that map to things the old notebook was doing manually or couldn't do at all:

- `handle_parsing_errors=True` (used on the old self-ask agent) → a `@wrap_tool_call` middleware that catches tool exceptions and returns a friendly `ToolMessage`
- Conversation memory getting too long → `SummarizationMiddleware`
- Requiring a human to approve a sensitive tool call → `HumanInTheLoopMiddleware`

See the [middleware guide](https://docs.langchain.com/oss/python/langchain/middleware/overview) for the full list of hooks (`before_agent`, `before_model`, `wrap_model_call`, `wrap_tool_call`, `after_model`, `after_agent`).

In [6]:
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

## Bonus: middleware — the real replacement for most `AgentType` behavior
@wrap_tool_call
def handle_tool_errors(request, handler):
    """Equivalent to the old `handle_parsing_errors=True` flag."""
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(
            content=f"Tool error: please rephrase your request. ({e})",
            tool_call_id=request.tool_call["id"],
        )


robust_agent = create_agent(
    model=model,
    tools=tools,
    middleware=[handle_tool_errors],
)

result = robust_agent.invoke(
    {"messages": [{"role": "user", "content": "Answer briefly from simple_qa; also check the web for any LangChain updates."}]}
)
print(result["messages"][-1].content)

[{'type': 'text', 'text': 'Could you please specify what question you would like me to answer briefly using simple_qa?\n\nIn the meantime, I will search for LangChain updates.', 'extras': {'signature': 'CuMEAWkUfRNnWtFX4S7afSLVsJpLF/ueEYtfX+gwoDwFX18zgD6Mq0W40B67VkvFdoSz+1jGiL2oAE4KTdprFtqdfamLNbQzm0RJftLr+cOYSeqk3mgWJXkzOlIq7az+6k+VsNdpo0SG+Q4zS5pJ9A7v+ZnhvnzXMmJvF8wn0QlxMZ+ChSTxHLicjH73S1oQ0/x9Qg+nJdhfjO09r31k+08YFQJi7TC+5k+ELZ6Ys6qvjQtc6P4fnHOND5uHdwHS2xJaeXQJlyS+gSr7d8Hs94LbXb7PG6UhD9QDrqSWx14ge7xu/UkZyUU4t+PE7YyotImd6P7HvFrZS7L0AT5U1DIJr4D3cejRJLAtZD1ybLY5lYp2qrYgvp3Me9SsXQ1z/uI1oyzxpEoaOfMoUmin+Aj5cbY08m1HALAJspXCvB/e+syMx8eAWGCLXBgYpkFDIbQIvnbQpxxbRuR/e9xhT7c9N327p8BYdooZrDJKUmOAiZ4LVaQ3QQ2I4IjViHJHIsfz6RCHFWdtWzupclKg9DkutfGbDGaeVMxWxSKO8xnaT/rS0tInkzxrQqC64huyrHhycIkw5Q3BZNKTso4Ozg65x3hGQgRn1PW8GatWHteG++DmEod5sXJCsZ/KuepRp3XYtC96qjQFBnmqd0rnRjkgdBY7p0/vPeW2m8OS4HDLaHn0tkmFqfHolDsXuErevyDc8dLfgK1yaphbhsNHsu+RK94wLCxxcRWd5wxqiOSJkuG6MnS6tXrSR5dDC3QcC44WLc1IHBhVCF04iNTX4N/14ISsO

---
### Reference diagrams (carried over from the original notebook)

The image cells below (`agenttypes1.png`, `agenttype2.png`, `pro1.png`, `pro2.png`, `pro3.png`) illustrated the legacy `AgentType` architectures in the original notebook. The image files themselves were **not part of the uploaded notebook**, so they won't render unless you place the original PNGs in the same folder as this notebook. Conceptually, the v1 replacement for all of these diagrams is the single **`create_agent` + middleware** graph loop described at the top of this notebook — one graph, customized with middleware, instead of seven separate agent classes.

## Summary

- `initialize_agent` + `AgentType` → **deprecated**, moved to `langchain-classic`
- `create_agent` is now the single entry point for building agents in LangChain v1
- Behavior differences between the 7 legacy agent types are now expressed via **prompting choices**, **tool schemas**, and **middleware** — not via separate agent classes
- Memory is handled by **checkpointers** (`InMemorySaver`, `PostgresSaver`, ...) + `thread_id`, not `ConversationBufferMemory`
- Function calling and parallel tool calls are the **default** behavior, not a special mode

For more detail, see the official release notes: <https://docs.langchain.com/oss/python/releases/langchain-v1#create_agent>